# Phase 2-plus — Step 2: Corpus Augmentation & Ingestion

## Objective
Scale the experimental corpus from 5 original documents (33 chunks) to a realistic corporate environment with **15-20 additional synthetic documents** spanning multiple departments, clearance levels, and PII types. Then re-run the Custom RBAC Chunking pipeline and populate ChromaDB + BM25 indexes.

## Why
Step 3 (Re-evaluating Dynamic RRF) needs a larger, noisier corpus where the static α=0.5 baseline will struggle and the intent-based router will show measurable improvements. The original 33-chunk corpus is too small and thematically homogeneous for this.

In [1]:
# Cell 1 — Imports & paths
import json, os, re, time
from dataclasses import dataclass
from typing import Optional
import numpy as np
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
import chromadb

BASE = os.path.dirname(os.path.abspath("__file__"))
RAW_DOCS_DIR = os.path.join(BASE, "..", "..", "data", "raw_docs")
RESULTS_DIR = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph2")
PH1_CHUNK_FILE = os.path.join(BASE, "..", "..", "data", "results", "notebook_results", "ph1", "chunk_results.json")

os.makedirs(RAW_DOCS_DIR, exist_ok=True)
print(f"Raw docs dir: {os.path.abspath(RAW_DOCS_DIR)}")
print(f"Existing docs: {os.listdir(RAW_DOCS_DIR)}")
print("Ready.")

Raw docs dir: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\data\raw_docs
Existing docs: ['clients-and-billings.xlsx', 'distribution-contract-2026.docx', 'server_logs_witty_backend.txt', 'Witty-Financial-Report-2025.pdf', 'Witty-QuickGuide-EN.pdf']
Ready.


## Cell 2 — Generate 18 Synthetic Corporate Documents

Documents span 6 departments with varying clearance levels and PII density:

| Category | Count | Clearance | PII Types |
|----------|-------|-----------|-----------|
| Financial Reports | 3 | 2-3 | Monetary, IBAN |
| HR Policies & Records | 3 | 2-3 | Names, emails, ID cards |
| IT Infrastructure Logs | 3 | 2 | IPs, passwords, server names |
| Client Contracts | 3 | 2-3 | Names, IBANs, SWIFT codes |
| Product Documentation | 3 | 0 | Public info, no PII |
| Internal Memos | 3 | 1-2 | Mixed PII, internal comms |

In [2]:
# Cell 2 — Generate 18 synthetic documents on disk
# Each document is a self-contained .txt or .md file with realistic corporate content.

SYNTHETIC_DOCS = []

def write_doc(filename: str, content: str, metadata: dict) -> dict:
    """Write a document to disk and register its metadata."""
    path = os.path.join(RAW_DOCS_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    entry = {"filename": filename, "path": path, "chars": len(content), **metadata}
    SYNTHETIC_DOCS.append(entry)
    return entry

# ═══════════════════════════════════════════════════════════
# FINANCIAL REPORTS (clearance 2-3, finance dept)
# ═══════════════════════════════════════════════════════════

write_doc("fin_q1_2026_revenue.txt", """MICROGATE S.R.L. — REVENUE REPORT Q1 2026
Department: Finance
Classification: CONFIDENTIAL

1. Quarterly Summary
Total gross revenue for Q1 2026 reached €5.1 million, a 21% increase year-over-year. The Witty Timer product line contributed 72% of total revenue, with hardware kit sales remaining the primary driver at €3.3 million.

2. Revenue by Region
European markets generated €3.8 million (75%), with Spain leading at €1.2 million thanks to the exclusive distribution agreement with Deportes de Alto Rendimiento S.A. North American markets contributed €0.9 million, and Asia-Pacific accounted for €0.4 million.

3. Operating Margins
Gross margin improved to 62% from 58% in Q4 2025, driven by reduced component costs for photocell manufacturing. The cost per unit for complete Witty Kits decreased from €340 to €315.

4. Accounts Receivable
Outstanding invoices total €420,000. The largest debtor is Centro de Alto Rendimiento (CLI-001) with €12,500 pending from the Premium tier service agreement. All payments are channeled through our primary account: IBAN: ES79 2100 0813 6101 2345 6789 (Banco Santander, Madrid branch).
""", {"clearance_level": 3, "allowed_departments": "finance", "doc_type": "report",
      "global_topic": "Q1 2026 revenue and financial performance"})

write_doc("fin_budget_forecast_2027.txt", """MICROGATE S.R.L. — BUDGET FORECAST 2027
Department: Finance — For Executive Review Only
Classification: STRICTLY CONFIDENTIAL

1. R&D Investment Plan
Total R&D allocation: €2.8 million (up from €1.5M in 2026). Primary focus areas include next-generation lithium batteries for photocells (€1.2M), Witty Manager software cloud migration (€0.9M), and IoT sensor integration (€0.7M).

2. Personnel Costs
Projected headcount increase from 45 to 58 employees. New hires concentrated in Engineering (8 positions) and Sales (5 positions). Average salary budget per employee: €52,000 annually.

3. Capital Expenditure
Server infrastructure upgrade: €180,000 for on-premise GPU cluster to support internal AI/ML workloads. This replaces the aging Dell PowerEdge R740 servers currently running production workloads at IP 10.0.1.50.

4. Treasury Management
Primary operating account remains IBAN: IT60 X054 2811 1010 0000 0123 456 (Banca Intesa, Bolzano). Secondary EUR account for Spanish operations: IBAN: ES79 2100 0813 6101 2345 6789. SWIFT code for international transfers: BCITITMM.
""", {"clearance_level": 3, "allowed_departments": "finance", "doc_type": "report",
      "global_topic": "2027 budget forecast and capital planning"})

write_doc("fin_expense_policy.md", """# Microgate S.R.L. — Corporate Expense Policy
**Version:** 2.1 | **Effective:** January 2026 | **Department:** Finance

## 1. Travel Expenses
All business travel must be pre-approved by the department head. Maximum daily allowances: €150 (EU), €200 (non-EU). Receipts required for all expenses above €25.

## 2. Equipment Purchases
IT equipment purchases above €500 require Finance department approval. Purchases above €5,000 require CFO signature. All hardware must be registered in the IT asset management system.

## 3. Reimbursement Process
Submit expense reports within 30 days of incurrence. Reimbursements are processed on the 15th of each month. Bank transfers are made to the employee's registered account.

## 4. Corporate Credit Cards
Corporate cards are issued to department heads and senior managers only. Monthly spending limit: €3,000 per card. Unauthorized personal use will result in immediate card cancellation and disciplinary action.

## 5. Audit Trail
All financial transactions are logged and subject to quarterly internal audit. External audit conducted annually by Deloitte Italia S.p.A. Contact: Mr. Andrea Rossi, arossi@deloitte.it.
""", {"clearance_level": 1, "allowed_departments": "all", "doc_type": "policy",
      "global_topic": "Corporate expense and reimbursement policy"})

# ═══════════════════════════════════════════════════════════
# HR POLICIES & RECORDS (clearance 2-3, hr dept)
# ═══════════════════════════════════════════════════════════

write_doc("hr_employee_directory.txt", """MICROGATE S.R.L. — EMPLOYEE DIRECTORY (Partial Extract)
Department: Human Resources
Classification: CONFIDENTIAL — HR ONLY

Engineering Department:
  EMP-001  Marco Bianchi      Lead Developer       marco.bianchi@microgate.it       DNI: 12345678X
  EMP-002  Sofia Fernandez    Backend Engineer     sofia.fernandez@microgate.it     DNI: 87654321Y
  EMP-003  Luca Moretti       DevOps Engineer      luca.moretti@microgate.it        DNI: 11223344Z

Sales Department:
  EMP-004  Isabella Romano    Sales Director       isabella.romano@microgate.it     DNI: 55667788W
  EMP-005  Diego Martinez     Account Manager      diego.martinez@microgate.it      DNI: 99887766V

Finance Department:
  EMP-006  Chiara Colombo     CFO                  chiara.colombo@microgate.it      DNI: 44556677T
  EMP-007  Roberto Esposito   Financial Analyst    roberto.esposito@microgate.it    DNI: 33445566R

Legal Department:
  EMP-008  Valentina Ricci    Legal Counsel        valentina.ricci@microgate.it     DNI: 22334455S
""", {"clearance_level": 3, "allowed_departments": "hr", "doc_type": "directory",
      "global_topic": "Employee directory with personal identification data"})

write_doc("hr_onboarding_guide.md", """# Microgate S.R.L. — New Employee Onboarding Guide
**Version:** 3.0 | **Last Updated:** March 2026

## Welcome to Microgate!
This guide will help you get started during your first week at the company.

## 1. First Day Checklist
- Report to reception at Via Waltraud Gebert Deeg, 3e, Bolzano at 9:00 AM
- Collect your employee badge and laptop from IT
- Complete tax forms (Modello 770) with HR
- Set up your corporate email (@microgate.it) and Slack workspace

## 2. IT Setup
Your workstation will be pre-configured with the following tools:
- Witty Manager Software (development build)
- Git, Docker, and VS Code (Engineering)
- SAP Business One (Finance)
- Salesforce CRM (Sales)

For VPN access, contact the IT helpdesk at helpdesk@microgate.it or extension 4500. Default VPN credentials will be provided separately via encrypted email.

## 3. Security Awareness
- Never share your credentials or access badges
- Report suspicious emails to security@microgate.it
- Lock your workstation when leaving your desk (Win+L)
- All company data is classified: Public (0), Internal (1), Confidential (2), or Strict (3)

## 4. Key Contacts
- HR Manager: D. Alessandra Fontana — afontana@microgate.it
- IT Helpdesk: helpdesk@microgate.it — Ext. 4500
- Building Security: security@microgate.it — Ext. 4100
""", {"clearance_level": 0, "allowed_departments": "all", "doc_type": "guide",
      "global_topic": "New employee onboarding procedures and contacts"})

write_doc("hr_salary_bands_2026.txt", """MICROGATE S.R.L. — SALARY BANDS & COMPENSATION STRUCTURE
Department: Human Resources
Classification: STRICTLY CONFIDENTIAL — HR MANAGEMENT ONLY

1. Engineering Department
   Junior Developer (0-2 yrs): €35,000 - €42,000
   Mid-Level Developer (2-5 yrs): €42,000 - €55,000
   Senior Developer (5+ yrs): €55,000 - €72,000
   Lead/Architect: €72,000 - €90,000

2. Sales Department
   Account Executive: €30,000 base + 15% commission
   Account Manager: €38,000 base + 12% commission
   Sales Director: €55,000 base + 10% commission + €5,000 annual bonus

3. Finance Department
   Financial Analyst: €36,000 - €48,000
   Senior Accountant: €48,000 - €60,000
   CFO: €95,000 + performance bonus up to 20%

4. Current Payroll Summary
   Total monthly payroll: €187,500 (45 employees)
   Annual payroll projection: €2,250,000
   Benefits overhead: 32% of base salary
   Employer social security contributions: processed via IBAN: IT60 X054 2811 1010 0000 0123 456
""", {"clearance_level": 3, "allowed_departments": "hr", "doc_type": "report",
      "global_topic": "Salary bands and compensation structure for all departments"})

# ═══════════════════════════════════════════════════════════
# IT INFRASTRUCTURE LOGS (clearance 2, engineering dept)
# ═══════════════════════════════════════════════════════════

write_doc("it_server_logs_march_2026.txt", """[2026-03-10 08:00:01] INFO: Daily system health check initiated on cluster node-01.
[2026-03-10 08:00:05] INFO: Database backup completed successfully. Size: 2.4 GB.
[2026-03-10 08:15:22] WARNING: Disk usage on /data partition reached 82% on node-02 (IP 10.0.1.52).
[2026-03-10 09:30:00] INFO: Scheduled maintenance window opened. Services entering graceful degradation.
[2026-03-10 09:31:15] INFO: ChromaDB vector store reindexing started. Collection: prod_documents (15,234 vectors).
[2026-03-10 09:45:00] SUCCESS: Reindexing completed in 13m 45s. No data loss detected.
[2026-03-10 10:12:33] ERROR: SSL certificate expiry warning for api.microgate.it. Expires: 2026-04-10.
[2026-03-10 11:00:00] INFO: Automated penetration test initiated from security scanner at 10.0.5.100.
[2026-03-10 11:05:44] WARNING: 3 failed SSH login attempts detected from IP 203.0.113.42 targeting admin@node-01.
[2026-03-10 11:06:00] INFO: IP 203.0.113.42 added to temporary blocklist (24h).
[2026-03-10 14:00:00] INFO: Maintenance window closed. All services restored to full capacity.
[2026-03-10 14:05:12] INFO: Post-maintenance health check: all 12 endpoints responding normally.
""", {"clearance_level": 2, "allowed_departments": "engineering", "doc_type": "log",
      "global_topic": "Server infrastructure logs for March 2026 maintenance window"})

write_doc("it_incident_report_IR2026_003.txt", """MICROGATE S.R.L. — INCIDENT REPORT IR-2026-003
Department: Engineering / IT Security
Classification: CONFIDENTIAL
Date: 2026-03-15

1. Incident Summary
A brute-force attack was detected against the Witty Manager API authentication endpoint. The attacker used a botnet originating from multiple IPs, primarily from IP 185.220.101.34 and IP 45.134.26.90.

2. Timeline
- 14:22 UTC: Anomalous login rate detected (>200 attempts/minute)
- 14:25 UTC: WAF rate limiting activated, blocking source IPs
- 14:30 UTC: Security team notified via PagerDuty
- 14:45 UTC: Manual review confirmed no successful breaches
- 15:00 UTC: Emergency password rotation initiated for all service accounts
- 15:10 UTC: New API master password set: 'mg_api_s3cur3_2026!@#'

3. Root Cause
The API endpoint /v2/auth/login lacked progressive rate limiting. The WAF caught it but only after 12,000 attempts. No credentials were compromised.

4. Remediation
- Implemented progressive rate limiting (5/10/30 second delays)
- Added CAPTCHA challenge after 3 failed attempts
- Rotated all service account passwords
- Filed report with Italian CERT (reference: CERT-IT-2026-0342)

5. Responsible Engineer: D. Luca Moretti (luca.moretti@microgate.it)
""", {"clearance_level": 2, "allowed_departments": "engineering", "doc_type": "report",
      "global_topic": "Security incident report for brute-force attack on API endpoint"})

write_doc("it_vpn_config_guide.md", """# Microgate VPN Configuration Guide
**Version:** 1.2 | **Classification:** Internal | **Department:** Engineering

## 1. Overview
All remote employees must connect via the corporate WireGuard VPN before accessing internal resources. The VPN gateway is hosted at vpn.microgate.it (IP: 93.184.216.34).

## 2. Client Setup
Download WireGuard from https://www.wireguard.com/install/. Import the configuration file provided by IT via encrypted email.

## 3. Network Topology
- Production subnet: 10.0.1.0/24
- Development subnet: 10.0.2.0/24
- Database cluster: 10.0.1.50 - 10.0.1.55
- ChromaDB instance: 10.0.1.60

## 4. Troubleshooting
If connection fails:
1. Verify your public key is registered with the VPN gateway
2. Check that UDP port 51820 is not blocked by your firewall
3. Contact helpdesk@microgate.it with your WireGuard client logs

## 5. Security Notes
- VPN sessions auto-expire after 12 hours of inactivity
- Split tunneling is disabled; all traffic routes through the VPN
- Access logs are retained for 90 days for audit purposes
""", {"clearance_level": 1, "allowed_departments": "engineering", "doc_type": "guide",
      "global_topic": "VPN configuration and network topology documentation"})

# ═══════════════════════════════════════════════════════════
# CLIENT CONTRACTS (clearance 2-3, legal/sales dept)
# ═══════════════════════════════════════════════════════════

write_doc("contract_federacion_atletismo.txt", """CONTRATO DE SUMINISTRO — MICROGATE S.R.L.
Referencia: CTR-2026-FED-002
Fecha: 15 de febrero de 2026

REUNIDOS
De una parte, Microgate S.R.L., con domicilio en Via Waltraud Gebert Deeg, 3e, Bolzano, Italia, representada por D. Giovanni Alberti (CEO), en adelante "EL PROVEEDOR".
De otra parte, Federación de Atletismo de España, con domicilio en Madrid, representada por D. Marcos Ruiz, en adelante "EL CLIENTE".

CLÁUSULAS

PRIMERA: Objeto del Contrato
EL PROVEEDOR suministrará 50 unidades del producto Witty Wireless Training Timer (kit completo) para equipar los centros de entrenamiento de la federación a nivel nacional.

SEGUNDA: Precio y Condiciones
El precio unitario acordado es de €780 netos por kit completo (descuento del 8.2% sobre tarifa estándar). Importe total del contrato: €39,000.

TERCERA: Condiciones de Pago
El pago se realizará en dos plazos: 50% a la firma del contrato y 50% a la entrega. Transferencia bancaria al IBAN: IT60 X054 2811 1010 0000 0123 456 SWIFT: BCITITMM.

CUARTA: Plazo de Entrega
La entrega se efectuará en un plazo máximo de 45 días naturales desde la firma.

QUINTA: Garantía
EL PROVEEDOR ofrece garantía de 24 meses sobre defectos de fabricación.
""", {"clearance_level": 2, "allowed_departments": "legal", "doc_type": "contract",
      "global_topic": "Supply contract with Spanish Athletics Federation for Witty Timers"})

write_doc("contract_universidad_deporte.txt", """ACUERDO DE COLABORACIÓN ACADÉMICA — MICROGATE S.R.L.
Referencia: CTR-2026-UNI-003
Fecha: 1 de marzo de 2026

REUNIDOS
De una parte, Microgate S.R.L., representada por D. Giovanni Alberti.
De otra parte, Universidad del Deporte, representada por Dra. Elena Torres, Directora del Departamento de Ciencias del Movimiento.

CLÁUSULAS

PRIMERA: Objeto
Colaboración académica para la investigación en biomecánica deportiva utilizando el sistema Witty Timer como herramienta de medición. La universidad recibirá 10 kits Witty Timer en régimen de cesión temporal (3 años).

SEGUNDA: Contraprestaciones
La universidad publicará los resultados de investigación citando a Microgate como proveedor tecnológico. Microgate tendrá acceso preferente a los datos de rendimiento (anonimizados) generados durante los estudios.

TERCERA: Confidencialidad
Ambas partes se comprometen a no revelar los términos económicos del acuerdo. Coste de cesión: €0 (valoración de mercado: €8,500 por kit).

CUARTA: Contacto Administrativo
Universidad: Dra. Elena Torres — etorres@unideporte.edu — Tel: +34 912 345 678
Microgate: D. Isabella Romano — isabella.romano@microgate.it
""", {"clearance_level": 2, "allowed_departments": "legal", "doc_type": "contract",
      "global_topic": "Academic collaboration agreement with Universidad del Deporte"})

write_doc("contract_fitplus_maintenance.txt", """SERVICE LEVEL AGREEMENT (SLA) — MICROGATE S.R.L.
Reference: SLA-2026-FIT-004
Date: March 20, 2026

PARTIES
Provider: Microgate S.R.L., Bolzano, Italy
Client: Gimnasios FitPlus S.L., Madrid, Spain
Client Contact: D. David Silva — dsilva@fitplus.com — CLI-004

1. Scope of Service
Annual maintenance and calibration service for 5 Witty Timer units deployed across FitPlus gym locations in Madrid, Barcelona, and Valencia.

2. Service Terms
- Quarterly on-site calibration visits
- 48-hour response time for critical hardware failures
- Remote software support via Witty Manager (Mon-Fri, 9:00-18:00 CET)
- Firmware updates included

3. Pricing
Annual service fee: €1,150 (Basic tier)
Payment terms: Annual invoice, due within 30 days
Bank transfer to: IBAN: IT60 X054 2811 1010 0000 0123 456

4. Penalties
Failure to meet 48-hour response SLA: 5% discount on next annual fee per incident.

5. Duration
This SLA is valid from April 1, 2026 to March 31, 2027, with automatic renewal.
""", {"clearance_level": 2, "allowed_departments": "sales", "doc_type": "contract",
      "global_topic": "Service level agreement with FitPlus gyms for Witty Timer maintenance"})

# ═══════════════════════════════════════════════════════════
# PRODUCT DOCUMENTATION (clearance 0, public)
# ═══════════════════════════════════════════════════════════

write_doc("product_witty_timer_specs.md", """# Witty Wireless Training Timer — Technical Specifications
**Product Code:** WTT-2026 | **Version:** 4.0

## Physical Characteristics
- Timer Unit: 145 x 95 x 35 mm, 280g
- Photocell: 120 x 80 x 45 mm, 195g (each)
- Operating temperature: -10°C to +50°C
- Water resistance: IP65

## Wireless Communication
- Protocol: Proprietary 2.4 GHz (Microgate MG-Link)
- Range: up to 150 meters line-of-sight
- Latency: < 1ms (timing accuracy: ±0.001s)
- Channels: 16 selectable (multi-lane support)

## Battery
- Timer: 3,200 mAh Li-Ion, up to 40 hours continuous use
- Photocell: 2,400 mAh Li-Ion, up to 30 hours continuous use
- Charging: USB-C, 2.5 hours full charge (both devices)

## Software Compatibility
- Witty Manager Desktop: Windows 10/11, macOS 12+
- Witty Manager Mobile: iOS 15+, Android 12+
- API: REST API available for integration (documentation at docs.microgate.it)
""", {"clearance_level": 0, "allowed_departments": "all", "doc_type": "manual",
      "global_topic": "Witty Timer technical specifications and hardware details"})

write_doc("product_photocell_alignment.md", """# Photocell Alignment & Calibration Guide
**Applicable to:** Witty Photocell v3.x and v4.x

## 1. Initial Setup
Mount the photocell on the telescopic tripod at a height between 80cm and 120cm. Position the reflector directly opposite at the same height, with a clear line of sight.

## 2. Alignment Procedure
1. Power on the photocell by pressing the ON button for 1 second
2. The status LED will blink green (battery OK) or orange (battery low)
3. Rotate the photocell slowly until the continuous beep stops — this indicates correct alignment
4. Lock the tripod head to fix the position

## 3. Calibration Verification
Run a test pass through the photocell beam. The timer should register a split time. If no time is recorded:
- Check reflector angle (must be perpendicular to beam)
- Clean the photocell lens with the included microfiber cloth
- Verify that the timer and photocell are on the same channel (Settings > Wireless > Channel)

## 4. Multi-Gate Configuration
For training courses with multiple gates (e.g., agility drills):
- Maximum 8 photocell pairs per timer
- Assign sequential gate numbers (Gate 1, Gate 2, etc.)
- Minimum distance between gates: 2 meters
- The timer automatically computes split times and total elapsed time

## 5. Troubleshooting
| Symptom | Cause | Solution |
|---------|-------|----------|
| No beep during alignment | Reflector misaligned | Adjust angle by ±5° |
| Intermittent timing | Obstruction in beam path | Clear debris/moisture |
| Short battery life | Firmware outdated | Update via Witty Manager |
""", {"clearance_level": 0, "allowed_departments": "all", "doc_type": "manual",
      "global_topic": "Photocell alignment, calibration, and multi-gate configuration"})

write_doc("product_release_notes_v4.txt", """WITTY MANAGER SOFTWARE — RELEASE NOTES v4.2.0
Release Date: March 2026

NEW FEATURES:
- Multi-athlete tracking: Support for up to 32 athletes in a single session
- Cloud sync: Optional synchronization with Microgate Cloud (requires subscription)
- Export to CSV/PDF: One-click export of training session data
- Dark mode: Available in Settings > Appearance

BUG FIXES:
- Fixed issue where split times > 99.999s would overflow the display
- Resolved Bluetooth connectivity drops on Windows 11 23H2
- Corrected unit conversion error in km/h display (was showing m/s * 3.5 instead of * 3.6)

KNOWN ISSUES:
- macOS Sonoma: Occasional USB detection delay (workaround: reconnect cable)
- Android 14: Notification permissions must be manually granted after install

SYSTEM REQUIREMENTS:
- Windows 10/11 (64-bit), macOS 12+, iOS 15+, Android 12+
- 200 MB disk space, 4 GB RAM minimum
- Bluetooth 4.0+ for wireless connection
- USB-C port for wired connection and charging
""", {"clearance_level": 0, "allowed_departments": "all", "doc_type": "changelog",
      "global_topic": "Witty Manager software v4.2.0 release notes and bug fixes"})

# ═══════════════════════════════════════════════════════════
# INTERNAL MEMOS (clearance 1-2, mixed departments)
# ═══════════════════════════════════════════════════════════

write_doc("memo_q2_sales_targets.txt", """INTERNAL MEMO — SALES TARGETS Q2 2026
From: D. Isabella Romano, Sales Director
To: Sales Team
Date: April 1, 2026
Classification: INTERNAL

Team,

Following our strong Q1 performance (€5.1M total revenue), management has set the following targets for Q2:

1. Hardware Sales Target: €3.8M (+15% vs Q1)
   - Focus on the Spanish market where our distribution agreement with Deportes de Alto Rendimiento S.A. gives us exclusive access
   - Target 3 new Premium-tier clients in addition to our existing base (CLI-001 through CLI-004)

2. Key Accounts to Prioritize:
   - CLI-001 Centro de Alto Rendimiento (Laura Gómez, lgomez@car.es) — upsell to 10 additional units
   - CLI-002 Federación de Atletismo (Marcos Ruiz) — contract renewal due June 2026
   - CLI-005 (NEW) Hospital Deportivo de Barcelona — initial demo scheduled April 15

3. Pricing Authority
   Maximum discount authority without CFO approval: 10% on orders > 20 units.
   For orders > €50,000, contact D. Chiara Colombo (CFO) directly.

Best regards,
Isabella Romano
isabella.romano@microgate.it
""", {"clearance_level": 1, "allowed_departments": "sales", "doc_type": "memo",
      "global_topic": "Internal sales targets and key account strategy for Q2 2026"})

write_doc("memo_it_security_reminder.txt", """INTERNAL MEMO — SECURITY AWARENESS REMINDER
From: IT Security Team
To: All Employees
Date: March 20, 2026
Classification: INTERNAL

Dear colleagues,

Following the recent security incident (ref: IR-2026-003), we remind all employees of the following mandatory security practices:

1. PASSWORD POLICY
   - Minimum 12 characters, including uppercase, lowercase, numbers, and symbols
   - Change passwords every 90 days
   - Never reuse passwords across systems
   - The emergency admin password for the backup recovery console has been rotated. Contact IT Security for the new credentials.

2. PHISHING AWARENESS
   - We detected 47 phishing attempts targeting @microgate.it addresses in March
   - Never click links in unexpected emails, even if they appear to come from colleagues
   - Report suspicious emails to security@microgate.it

3. PHYSICAL SECURITY
   - Always lock your workstation when leaving your desk
   - Do not allow tailgating through badge-controlled doors
   - Visitor badges must be worn visibly at all times

4. DATA CLASSIFICATION REMINDER
   Level 0 (Public): Product documentation, marketing materials
   Level 1 (Internal): Company policies, internal memos
   Level 2 (Confidential): Client contracts, server logs, financial data
   Level 3 (Strict): Passwords, banking details (IBAN/SWIFT), personal ID numbers

For questions, contact: D. Luca Moretti, luca.moretti@microgate.it

IT Security Team
""", {"clearance_level": 1, "allowed_departments": "all", "doc_type": "memo",
      "global_topic": "Company-wide security awareness reminder after incident"})

write_doc("memo_board_meeting_minutes.txt", """CONFIDENTIAL — BOARD MEETING MINUTES
Date: March 28, 2026
Attendees: D. Giovanni Alberti (CEO), D. Chiara Colombo (CFO), D. Isabella Romano (Sales), D. Marco Bianchi (CTO)

1. Financial Review
D. Colombo presented Q1 results: €5.1M revenue, 62% gross margin. Board approved the 2027 budget of €2.8M for R&D. The CFO flagged that our Spanish distributor's payment (€39,000 from Federación de Atletismo contract) is overdue by 15 days. Action: D. Romano to follow up with D. Marcos Ruiz.

2. Product Strategy
D. Bianchi proposed accelerating the IoT integration roadmap. The board approved hiring 3 additional embedded systems engineers. Target: IoT-enabled photocells ready for beta testing by Q4 2026.

3. Legal Update
The distribution exclusivity agreement with Deportes de Alto Rendimiento S.A. (D. Carlos Mendoza) expires in December 2026. Board authorized D. Alberti to begin renegotiation, with a target of extending exclusivity to 5 years in exchange for minimum purchase commitments of 200 units/year.

4. HR Matters
Employee satisfaction survey results: 78% positive (up from 71%). Two departures in Engineering to be replaced. Salary band review scheduled for June 2026.

5. Next Meeting: April 25, 2026

Minutes recorded by: D. Valentina Ricci (valentina.ricci@microgate.it)
""", {"clearance_level": 3, "allowed_departments": "finance", "doc_type": "minutes",
      "global_topic": "Board meeting minutes with strategic decisions and financial data"})

# ═══════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print(f"  SYNTHETIC DOCUMENT GENERATION COMPLETE")
print(f"{'='*70}")
print(f"  Total documents generated: {len(SYNTHETIC_DOCS)}")
print(f"  Total characters written:  {sum(d['chars'] for d in SYNTHETIC_DOCS):,}")
print()
for d in SYNTHETIC_DOCS:
    print(f"  {d['filename']:45s}  cl={d['clearance_level']}  dept={d['allowed_departments']:15s}  {d['chars']:>5,} chars")


  SYNTHETIC DOCUMENT GENERATION COMPLETE
  Total documents generated: 18
  Total characters written:  20,514

  fin_q1_2026_revenue.txt                        cl=3  dept=finance          1,129 chars
  fin_budget_forecast_2027.txt                   cl=3  dept=finance          1,077 chars
  fin_expense_policy.md                          cl=1  dept=all              1,159 chars
  hr_employee_directory.txt                      cl=3  dept=hr                 999 chars
  hr_onboarding_guide.md                         cl=0  dept=all              1,297 chars
  hr_salary_bands_2026.txt                       cl=3  dept=hr                 957 chars
  it_server_logs_march_2026.txt                  cl=2  dept=engineering      1,164 chars
  it_incident_report_IR2026_003.txt              cl=2  dept=engineering      1,214 chars
  it_vpn_config_guide.md                         cl=1  dept=engineering      1,043 chars
  contract_federacion_atletismo.txt              cl=2  dept=legal            1,193 chars

In [3]:
# Cell 3 — Custom RBAC Chunking Pipeline (replicated from Phase 1)

SENSITIVE_PATTERNS: list[dict] = [
    {"name": "password", "pattern": r"(?i)(?:password|contrase[ñn]a|pwd|override[_ ]?password)\s*[:=]\s*['\"]?[^\s'\"]+['\"]?", "escalate_to": 3},
    {"name": "iban", "pattern": r"(?:IBAN|Cuenta)[:\s]*[A-Z]{2}\d{2}[\s]?[A-Z0-9\s]{15,30}", "escalate_to": 3},
    {"name": "swift", "pattern": r"SWIFT[:\s]*[A-Z]{4}[A-Z]{2}[A-Z0-9]{2,5}", "escalate_to": 3},
    {"name": "email_personal", "pattern": r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", "escalate_to": 2},
    {"name": "ip_address", "pattern": r"\b(?:from IP|IP)\s*:?\s*\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b", "escalate_to": 2},
    {"name": "client_id", "pattern": r"CLI-\d{3,}", "escalate_to": 2},
    {"name": "person_name", "pattern": r"(?:D\.|Mr\.|Mrs\.|Dra?\.|Contacto:?)\s+[A-Z][a-záéíóúñ]+\s+[A-Z][a-záéíóúñ]+", "escalate_to": 2},
    {"name": "monetary_confidential", "pattern": r"(?:coste|cost|precio|price|margen|margin|salary|salario)[^.]*€\s?[\d.,]+", "escalate_to": 2},
    {"name": "dni", "pattern": r"DNI:\s*\d{7,8}[A-Z]", "escalate_to": 3},
    {"name": "emp_id", "pattern": r"EMP-\d{3,}", "escalate_to": 2},
]

STRUCTURAL_SEPARATORS = [
    r"\n(?=\d+\.\s+[A-Z])",
    r"\n(?=PRIMERA|SEGUNDA|TERCERA|CUARTA|QUINTA|CL[ÁA]USULAS)",
    r"\n(?=\[\d{4}-)",
    r"\n(?=Sheet:)",
    r"\n(?=#{1,3}\s)",
    r"\n\n",
]

def structural_split(text: str, min_chunk_size: int = 80) -> list[str]:
    best_chunks = [text]
    for pattern in STRUCTURAL_SEPARATORS:
        parts = re.split(pattern, text)
        parts = [p.strip() for p in parts if p.strip()]
        if len(parts) > 1:
            merged = []
            buffer = ""
            for part in parts:
                if buffer and len(buffer) >= min_chunk_size:
                    merged.append(buffer)
                    buffer = part
                else:
                    buffer = (buffer + "\n" + part).strip() if buffer else part
            if buffer:
                merged.append(buffer)
            if len(merged) > len(best_chunks):
                best_chunks = merged
    return best_chunks

def find_sensitive_spans(text: str) -> list[dict]:
    spans = []
    for sp in SENSITIVE_PATTERNS:
        for match in re.finditer(sp["pattern"], text):
            start = text.rfind("\n", 0, match.start())
            start = start + 1 if start != -1 else match.start()
            end = text.find("\n", match.end())
            end = end if end != -1 else match.end()
            spans.append({
                "type": sp["name"], "start": start, "end": end,
                "escalate_to": sp["escalate_to"], "text": text[start:end],
            })
    spans.sort(key=lambda x: x["start"])
    return spans

def custom_rbac_chunk(text: str, base_metadata: dict, min_chunk_size: int = 80) -> list[Document]:
    structural_parts = structural_split(text, min_chunk_size=min_chunk_size)
    result_chunks: list[Document] = []

    for part in structural_parts:
        sensitive_spans = find_sensitive_spans(part)

        if not sensitive_spans:
            meta = base_metadata.copy()
            meta["contains_PII"] = False
            meta["sensitivity_types"] = []
            result_chunks.append(Document(page_content=part, metadata=meta))
        else:
            used_ranges: list[tuple[int, int]] = []
            for span in sensitive_spans:
                meta = base_metadata.copy()
                meta["contains_PII"] = True
                meta["sensitivity_types"] = [span["type"]]
                meta["clearance_level"] = max(meta.get("clearance_level", 0), span["escalate_to"])
                result_chunks.append(Document(page_content=span["text"], metadata=meta))
                used_ranges.append((span["start"], span["end"]))

            used_ranges.sort()
            merged_ranges: list[tuple[int, int]] = []
            for start, end in used_ranges:
                if merged_ranges and start <= merged_ranges[-1][1]:
                    merged_ranges[-1] = (merged_ranges[-1][0], max(merged_ranges[-1][1], end))
                else:
                    merged_ranges.append((start, end))

            prev_end = 0
            remainder_parts = []
            for start, end in merged_ranges:
                if start > prev_end:
                    fragment = part[prev_end:start].strip()
                    if fragment:
                        remainder_parts.append(fragment)
                prev_end = end
            if prev_end < len(part):
                fragment = part[prev_end:].strip()
                if fragment:
                    remainder_parts.append(fragment)

            if remainder_parts:
                remainder_text = "\n".join(remainder_parts)
                meta = base_metadata.copy()
                meta["contains_PII"] = False
                meta["sensitivity_types"] = []
                result_chunks.append(Document(page_content=remainder_text, metadata=meta))

    for i, chunk in enumerate(result_chunks):
        chunk.metadata["chunk_id"] = f"aug_{i:03d}"

    return result_chunks

print("Custom RBAC chunking pipeline loaded.")

Custom RBAC chunking pipeline loaded.


In [4]:
# Cell 4 — Ingest: chunk all synthetic docs + original Ph1 corpus

# Load original Phase 1 chunks
with open(PH1_CHUNK_FILE, "r", encoding="utf-8") as f:
    ph1_data = json.load(f)

original_corpus: list[Document] = []
for src_file, chunks in ph1_data["custom_rbac"].items():
    for ch in chunks:
        meta = ch["metadata"].copy()
        meta.setdefault("contains_PII", False)
        meta.setdefault("sensitivity_types", [])
        original_corpus.append(Document(page_content=ch["page_content"], metadata=meta))
print(f"Original Phase 1 corpus: {len(original_corpus)} chunks")

# Chunk the 18 new synthetic documents
augmented_chunks: list[Document] = []
chunk_stats = []

for doc_info in SYNTHETIC_DOCS:
    filepath = doc_info["path"]
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    base_meta = {
        "source": filepath,
        "source_file": doc_info["filename"],
        "doc_type": doc_info["doc_type"],
        "global_topic": doc_info["global_topic"],
        "clearance_level": doc_info["clearance_level"],
        "allowed_departments": doc_info["allowed_departments"],
        "page_number": 1,
    }

    chunks = custom_rbac_chunk(text, base_meta)
    augmented_chunks.extend(chunks)

    pii_count = sum(1 for c in chunks if c.metadata.get("contains_PII"))
    pii_types = set()
    for c in chunks:
        st = c.metadata.get("sensitivity_types", [])
        if isinstance(st, list):
            pii_types.update(st)
    chunk_stats.append({
        "filename": doc_info["filename"],
        "total_chunks": len(chunks),
        "pii_chunks": pii_count,
        "pii_types": ", ".join(sorted(pii_types)) if pii_types else "-",
        "clearance": doc_info["clearance_level"],
        "dept": doc_info["allowed_departments"],
    })

# Reassign globally unique chunk IDs for augmented set
for i, chunk in enumerate(augmented_chunks):
    chunk.metadata["chunk_id"] = f"aug_{i:03d}"

# Combine: original + augmented
full_corpus = original_corpus + augmented_chunks

print(f"\nAugmented chunks: {len(augmented_chunks)} (from {len(SYNTHETIC_DOCS)} new docs)")
print(f"Full corpus: {len(original_corpus)} original + {len(augmented_chunks)} augmented = {len(full_corpus)} total")

print(f"\n{'='*85}")
print(f"  PER-DOCUMENT CHUNK BREAKDOWN")
print(f"{'='*85}")
stats_df = pd.DataFrame(chunk_stats)
print(stats_df.to_string(index=False))
print(f"\n  Augmented totals: {stats_df['total_chunks'].sum()} chunks, "
      f"{stats_df['pii_chunks'].sum()} with PII")

Original Phase 1 corpus: 33 chunks



Augmented chunks: 195 (from 18 new docs)
Full corpus: 33 original + 195 augmented = 228 total

  PER-DOCUMENT CHUNK BREAKDOWN
                         filename  total_chunks  pii_chunks                                       pii_types  clearance        dept
          fin_q1_2026_revenue.txt             8           3          client_id, iban, monetary_confidential          3     finance
     fin_budget_forecast_2027.txt             9           4         iban, ip_address, monetary_confidential          3     finance
            fin_expense_policy.md             8           2                     email_personal, person_name          1         all
        hr_employee_directory.txt            29          24                     dni, email_personal, emp_id          3          hr
           hr_onboarding_guide.md            13           6                     email_personal, person_name          0         all
         hr_salary_bands_2026.txt             6           1                            


  Augmented totals: 195 chunks, 85 with PII


In [5]:
# Cell 5 — Populate ChromaDB + BM25 with the full corpus

# --- Embedding model ---
print("Loading embedding model...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={"device": "cpu"})

# --- ChromaDB ---
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="augmented_corpus", metadata={"hnsw:space": "cosine"})

def sanitize(meta: dict) -> dict:
    return {k: (json.dumps(v) if isinstance(v, list) else v if isinstance(v, (str, int, float, bool)) else str(v))
            for k, v in meta.items()}

print(f"Computing embeddings for {len(full_corpus)} chunks...")
t0 = time.time()
texts = [d.page_content for d in full_corpus]
embs = embedding_model.embed_documents(texts)
embed_time = time.time() - t0
print(f"  Embeddings computed in {embed_time:.1f}s ({embed_time/len(full_corpus)*1000:.0f}ms per chunk)")

collection.add(
    documents=texts,
    embeddings=embs,
    ids=[f"c{i:04d}" for i in range(len(full_corpus))],
    metadatas=[sanitize(d.metadata) for d in full_corpus]
)
print(f"  ChromaDB populated: {collection.count()} documents")

# --- BM25 ---
def tokenize(t: str) -> list[str]:
    return re.findall(r"\w+", t.lower())

tok_corpus = [tokenize(d.page_content) for d in full_corpus]
bm25 = BM25Okapi(tok_corpus)
print(f"  BM25 index built: {len(tok_corpus)} documents")

# --- Corpus composition summary ---
cl_counts = {}
dept_counts = {}
pii_total = 0
for d in full_corpus:
    cl = d.metadata.get("clearance_level", 0)
    dept = d.metadata.get("allowed_departments", "all")
    cl_counts[cl] = cl_counts.get(cl, 0) + 1
    dept_counts[dept] = dept_counts.get(dept, 0) + 1
    if d.metadata.get("contains_PII"):
        pii_total += 1

print(f"\n{'='*60}")
print(f"  AUGMENTED CORPUS SUMMARY")
print(f"{'='*60}")
print(f"  Total chunks:       {len(full_corpus)}")
print(f"  Original (Ph1):     {len(original_corpus)}")
print(f"  New (augmented):    {len(augmented_chunks)}")
print(f"  Chunks with PII:    {pii_total}")
print(f"\n  By clearance level:")
for cl in sorted(cl_counts.keys()):
    print(f"    cl={cl}: {cl_counts[cl]} chunks")
print(f"\n  By department:")
for dept in sorted(dept_counts.keys()):
    print(f"    {dept}: {dept_counts[dept]} chunks")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Computing embeddings for 228 chunks...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  Embeddings computed in 3.1s (14ms per chunk)
  ChromaDB populated: 228 documents
  BM25 index built: 228 documents

  AUGMENTED CORPUS SUMMARY
  Total chunks:       228
  Original (Ph1):     33
  New (augmented):    195
  Chunks with PII:    100

  By clearance level:
    cl=0: 24 chunks
    cl=1: 25 chunks
    cl=2: 92 chunks
    cl=3: 87 chunks

  By department:
    all: 49 chunks
    engineering: 40 chunks
    finance: 36 chunks
    hr: 35 chunks
    legal: 32 chunks
    sales: 36 chunks


In [6]:
# Cell 6 — Export augmented corpus + metadata for Step 3

export = {
    "step": "Phase 3 - Step 2: Corpus Augmentation",
    "original_docs": 5,
    "synthetic_docs": len(SYNTHETIC_DOCS),
    "total_docs": 5 + len(SYNTHETIC_DOCS),
    "original_chunks": len(original_corpus),
    "augmented_chunks": len(augmented_chunks),
    "total_chunks": len(full_corpus),
    "pii_chunks": pii_total,
    "clearance_distribution": {str(k): v for k, v in sorted(cl_counts.items())},
    "department_distribution": dict(sorted(dept_counts.items())),
    "synthetic_doc_manifest": SYNTHETIC_DOCS,
    "per_document_stats": chunk_stats,
}

json_path = os.path.join(RESULTS_DIR, "ph2_plus_step2_augmented_corpus.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2, ensure_ascii=False, default=str)
print(f"Exported: {json_path}")

# Also export the full augmented chunk data for reuse in Step 3
augmented_export = {}
for d in full_corpus:
    src = d.metadata.get("source_file", "unknown")
    if src not in augmented_export:
        augmented_export[src] = []
    augmented_export[src].append({
        "page_content": d.page_content,
        "metadata": {k: v for k, v in d.metadata.items()}
    })

chunk_path = os.path.join(RESULTS_DIR, "ph2_plus_augmented_chunks.json")
with open(chunk_path, "w", encoding="utf-8") as f:
    json.dump(augmented_export, f, indent=2, ensure_ascii=False, default=str)
print(f"Exported: {chunk_path}")

print(f"\n{'='*60}")
print(f"  Step 2 COMPLETE")
print(f"{'='*60}")
print(f"  Files on disk:    {5 + len(SYNTHETIC_DOCS)} ({5} original + {len(SYNTHETIC_DOCS)} synthetic)")
print(f"  ChromaDB:         {collection.count()} vectors indexed")
print(f"  BM25:             {len(tok_corpus)} documents indexed")
print(f"  Total corpus:     {len(full_corpus)} chunks")
print(f"\n  Awaiting approval before Step 3.")

Exported: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\notebooks\ph2-retrieval-strategy\..\..\data\results\notebook_results\ph2\ph3_step2_augmented_corpus.json
Exported: D:\URV\TFG\ai-rag-context-auth-system\strategy-analysis\notebooks\ph2-retrieval-strategy\..\..\data\results\notebook_results\ph2\ph3_augmented_chunks.json

  Step 2 COMPLETE
  Files on disk:    23 (5 original + 18 synthetic)
  ChromaDB:         228 vectors indexed
  BM25:             228 documents indexed
  Total corpus:     228 chunks

  Awaiting approval before Step 3.
